# Topology optimization of substrate geometry
Loads the trained surrogate from saved weights, runs gradient-based topology
optimization (`optimize_support_map`), then applies the post-processing cleanup
pipeline to produce the final manufacturable binary support map.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
import matplotlib.pyplot as plt
from scipy.ndimage import convolve, label

from surrogate_model import UNet
from topology_loss_functions import (
    circular_density_filter_torch,
    tv_loss,
    connectivity_loss,
    binarization_loss,
)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

## Load the trained surrogate from saved weights

In [ ]:
# TODO: set the path to your saved checkpoint (.pth) from 02_train_surrogate.ipynb
CHECKPOINT_PATH = 'unet5_r2_0983_256_3007.pth'

model = UNet()
state_dict = torch.load(CHECKPOINT_PATH, weights_only=True)
model.load_state_dict(state_dict)
model = model.to(device)
model.eval()

## Topology optimization loop

In [ ]:
def optimize_support_map(model, support_map, target_i, target_j, Lx, Ly, n_steps=1000):
    model.eval()
    losses = []
    losses_physical = []   # loss_ratio + loss_abs (weighted, as actually used in the total loss)
    losses_sym = []        # loss_sym (weighted)
    losses_fill = []       # fill_loss (weighted)
    losses_ratio_raw = []  # raw loss_ratio (unweighted), useful for diagnostics
    losses_abs_raw = []    # raw loss_abs (unweighted)
    losses_sym_raw = []    # raw loss_sym (unweighted)
    losses_fill_raw = []   # raw fill_loss (unweighted)

    H, W = support_map.shape[-2], support_map.shape[-1]
    support_map = support_map.detach()
    xs = torch.linspace(0, Lx, W)
    ys = torch.linspace(0, Ly, H)
    grid_y, grid_x = torch.meshgrid(ys, xs, indexing='ij')
    grid_x = grid_x.unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
    grid_y = grid_y.unsqueeze(0).unsqueeze(0)  # (1,1,H,W)

    with torch.no_grad():
        dummy_out = model(torch.zeros(1, 3, H, W).to(device)).cpu()
        H_out, W_out = dummy_out.shape[-2], dummy_out.shape[-1]

    mask_waveguide = torch.ones(1, 1, H, W)
    c = H // 2
    mask_waveguide[0, 0, c - 40 : c + 40, :] = 0
    mask_traget = torch.zeros(1, 1, H_out, W_out)
    c = W // 2
    mask_traget[0, 0, c - 15 : c + 15, c-25:c+25] = 1
    mask_not_target = torch.zeros(1, 1, H_out, W_out)
    mask_not_target[0, 0, c - 15 : c + 15, :] = 1        # вся полоса по Y
    mask_not_target = mask_not_target * (1 - mask_traget)
    ti = int(target_i * H_out / H)
    tj = int(target_j * W_out / W)
    mask2 = torch.zeros(1, 1, H_out, W_out)
    mask2[0, 0, max(0, ti-50):min(H_out, ti+50), max(0, tj-50):min(W_out, tj+50)] = 1
    mask2_inv = 1 - mask2

    yy, xx = torch.meshgrid(torch.arange(H_out, dtype=torch.float32), torch.arange(W_out, dtype=torch.float32), indexing="ij",)
    cy, cx = H_out // 2, W_out // 2
    R = 150
    sigma = R / 2
    gauss_base = torch.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * sigma**2))
    gauss_base = gauss_base.unsqueeze(0).unsqueeze(0)
    gauss_base = gauss_base.to(device)

    existing = torch.clamp(support_map * mask_waveguide, 1e-4, 1 - 1e-4)
    delta = torch.logit(existing).detach().requires_grad_(True)

    optimizer = torch.optim.Adamax([delta], lr=1)
    n_free_pixels = mask_waveguide.sum()

    for i in range(n_steps):
        optimizer.zero_grad()
        w_tv = 0.5 + 0.09 * (i / n_steps)
        w_conn = 1
        w_bin = 0.5 + 0.08 * (i / n_steps)

        supp_soft = torch.sigmoid(delta)
        supp_soft = torch.sigmoid(delta)

        min_feature_size_px = 32
        r_filter_px = max(1.0, min_feature_size_px / 2.0)

        supp_filtered = circular_density_filter_torch(supp_soft, r_filter_px=r_filter_px,)

        supp_clip = supp_filtered * mask_waveguide + support_map * (1 - mask_waveguide)
        model_input = torch.cat([supp_clip, grid_x, grid_y], dim=1)  # (1,3,H,W)
        pred_strain = model(model_input.to(device)).cpu()
        strain_target = pred_strain * mask_traget
        strain_other = pred_strain * mask_not_target
        supp = supp_clip
        H2 = H // 2
        top = supp[..., :H2, :]
        bot = supp[..., H - H2:, :].flip(-2)
        loss_sym_y = F.mse_loss(top, bot)
        W2 = W // 2
        left = supp[..., :W2]
        right = supp[..., W - W2:].flip(-1)
        loss_sym_x = F.mse_loss(left, right)

        loss_sym = loss_sym_x + loss_sym_y

        eps = 1e-8
        mean_target = strain_target.sum() / (mask_traget.sum() + eps)
        mean_other = strain_other.sum() / (mask_not_target.sum() + eps)
        fill_factor = (supp_clip * mask_waveguide).sum() / n_free_pixels
        fill_loss = (fill_factor - 0.005)**2
        loss_ratio = -(torch.log(mean_target + eps) - torch.log(mean_other + eps))
        loss_abs = -mean_target
        loss_tv = tv_loss(supp_clip)
        loss_conn = connectivity_loss(supp_clip, min_neighbors=2)
        loss_bin = binarization_loss(supp_clip)
        max_in_target = (pred_strain * mask_traget).max()
        gauss_target = gauss_base * max_in_target
        gauss_mask = (gauss_base > 1e-3).float().cpu()
        loss_gauss = F.mse_loss(pred_strain * gauss_mask, gauss_target.cpu() * gauss_mask)
        loss_max = (torch.max(strain_target) - torch.max(pred_strain))**2

        if i < 600:
            loss = 1000*loss_ratio + 10000*loss_abs + 0 * loss_max
        elif i < 700:
            loss = 1000*loss_ratio + 10000*loss_abs + 0*loss_tv + 0*fill_loss + 1000*loss_sym + 0 * loss_max
        else:
            loss = 1000*loss_ratio + 0*loss_abs + 0*loss_tv + 0*loss_bin + 1000*fill_loss + 1000*loss_sym + 0*loss_gauss + 0 * loss_max

        loss.backward()
        optimizer.step()
        losses.append(loss.detach().item())

        # ---- extract weighted component contributions actually present in `loss` above ----
        if i < 600:
            physical_term = 1000*loss_ratio + 10000*loss_abs
            sym_term = torch.tensor(0.0)
            fill_term = torch.tensor(0.0)
        elif i < 700:
            physical_term = 1000*loss_ratio + 10000*loss_abs
            sym_term = 1000*loss_sym
            fill_term = torch.tensor(0.0)
        else:
            physical_term = 1000*loss_ratio
            sym_term = 1000*loss_sym
            fill_term = 1000*fill_loss

        losses_physical.append(physical_term.detach().item())
        losses_sym.append(sym_term.detach().item())
        losses_fill.append(fill_term.detach().item())
        losses_ratio_raw.append(loss_ratio.detach().item())
        losses_abs_raw.append(loss_abs.detach().item())
        losses_sym_raw.append(loss_sym.detach().item())
        losses_fill_raw.append(fill_loss.detach().item())

        if i % 50 == 0:
            print(f'[{i:4d}] ratio={loss_ratio.item():.4f} abs={loss_abs.item():.5f} total={loss.item():.4f}')
            plt.clf(); plt.plot(losses); plt.grid(True)
            free_result = (torch.sigmoid(delta).detach().cpu() > 0.5).float() * mask_waveguide
            result = (free_result + support_map * (1 - mask_waveguide)).squeeze().numpy()

    free_result = (torch.sigmoid(delta).detach().cpu() > 0.5).float() * mask_waveguide
    result = (free_result + support_map * (1 - mask_waveguide)).squeeze().numpy()

    np.savez_compressed(
        'topopt_loss_history.npz',
        total=np.array(losses, dtype=np.float32),
        physical=np.array(losses_physical, dtype=np.float32),
        symmetry=np.array(losses_sym, dtype=np.float32),
        fill=np.array(losses_fill, dtype=np.float32),
        ratio_raw=np.array(losses_ratio_raw, dtype=np.float32),
        abs_raw=np.array(losses_abs_raw, dtype=np.float32),
        symmetry_raw=np.array(losses_sym_raw, dtype=np.float32),
        fill_raw=np.array(losses_fill_raw, dtype=np.float32),
    )

    return np.clip(result, 0, 1), losses

## Run the optimization
Load the fixed photonic platform (resonant waveguide geometry) and the target
zone coordinates, then call `optimize_support_map`.

In [ ]:
# load the SWG support map
support_map = np.load('swg_256.npz')['array']
x = torch.tensor(support_map, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

H_orig, W_orig = x.shape[-2], x.shape[-1]
pad_h = (4 - H_orig % 4) % 4
pad_w = (4 - W_orig % 4) % 4
c = H_orig // 2

# Set the central part as a non-changeable region
mask = torch.zeros_like(x)
mask[0, 0, c - 40: c + 40, :] = 1
inv_mask = 1.0 - mask

x_padded = F.pad(x * mask, (0, pad_w, 0, pad_h))

print(f"shape: {x_padded.shape}, range: [{x_padded.min():.4f}, {x_padded.max():.4f}]")

In [ ]:
optimized_sup_map,  loss_opti = optimize_support_map(model, x_padded, 128, 128, 12,12, 2000)

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

Lx, Ly = 12, 12
ext = [0, Lx, 0, Ly]

opt_tensor = torch.tensor(optimized_sup_map, dtype=torch.float32)

xs = torch.linspace(0, Lx, opt_tensor.shape[1])
ys = torch.linspace(0, Ly, opt_tensor.shape[0])
grid_y, grid_x = torch.meshgrid(ys, xs, indexing='ij')

x_opt = torch.cat([
    opt_tensor.unsqueeze(0).unsqueeze(0),
    grid_x.unsqueeze(0).unsqueeze(0),
    grid_y.unsqueeze(0).unsqueeze(0),
], dim=1)  # (1,3,H,W)

H, W = opt_tensor.shape
pad_h = (16 - H % 16) % 16
pad_w = (16 - W % 16) % 16
x_opt_pad = F.pad(x_opt, (0, pad_w, 0, pad_h))

model.eval()
with torch.no_grad():
    pred_opt = model(x_opt_pad.to(device)).cpu()[0, 0, :H, :W].numpy()

# visualisation 
mm = 1 / 25.4
fig, axes = plt.subplots(1, 2, figsize=(120 * mm, 60 * mm), constrained_layout=True)

# material density map
ax = axes[0]
im0 = ax.imshow(optimized_sup_map, cmap='gray_r', origin='lower',
                extent=ext, aspect='equal', vmin=0, vmax=1)
divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='5%', pad=0.04)
cb = plt.colorbar(im0, cax=cax, ticks=[0, 1])
cb.set_label('Material', fontsize=6)
cb.ax.set_yticklabels(['0', '1'])
cb.ax.tick_params(labelsize=5.5, width=0.5, length=2)
cb.outline.set_linewidth(0.5)
ax.set_title('Optimized support map', fontsize=7, pad=3)
ax.set_xlabel('μm', fontsize=6)
ax.set_ylabel('μm', fontsize=6)
ax.tick_params(labelsize=6, top=False, right=False)

# predicted strain
ax = axes[1]
vmax = np.abs(pred_opt).max()
im1 = ax.imshow(pred_opt, cmap='coolwarm', origin='lower',
                extent=ext, aspect='equal', vmin=-vmax, vmax=vmax)
divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='5%', pad=0.04)
cb = plt.colorbar(im1, cax=cax)
cb.set_label('Strain', fontsize=6)
cb.ax.tick_params(labelsize=5.5, width=0.5, length=2)
cb.outline.set_linewidth(0.5)
ax.set_title('Predicted strain', fontsize=7, pad=3)
ax.set_xlabel('μm', fontsize=6)
ax.tick_params(labelsize=6, top=False, right=False, labelleft=False)

# plt.savefig('optimized_result.png', dpi=300, bbox_inches='tight')
plt.show()

## Post-processing cleanup
Smooths the continuous density map, binarizes it, removes sub-resolution
features and holes, and restores the protected (waveguide) region.

In [ ]:
def circular_density_filter_np(pred_opt, r_filter, distance_between_elements=1.0):
    pred_opt = np.asarray(pred_opt, dtype=np.float64)
    filter_size_elements = int(2 * (r_filter / distance_between_elements) + 1)
    if filter_size_elements % 2 == 0:
        filter_size_elements += 1
    half = filter_size_elements // 2

    y, x = np.ogrid[-half:half+1, -half:half+1]
    distances = (x**2 + y**2) * distance_between_elements**2
    mask = distances <= r_filter**2

    weights = r_filter - np.sqrt(distances)
    weights = np.where(mask, weights, 0.0)
    weights = np.clip(weights, 0, None)

    denom = weights.sum()
    if denom == 0:
        return pred_opt.copy()
    return convolve(pred_opt, weights, mode='nearest') / denom


def remove_small_objects_scipy(binary_mask, min_size_px, connectivity=2):
    binary_mask = binary_mask.astype(bool)
    structure = np.ones((3, 3), dtype=int) if connectivity == 2 else np.array([[0,1,0],[1,1,1],[0,1,0]])
    labeled, n_features = label(binary_mask, structure=structure)
    if n_features == 0:
        return binary_mask.copy()
    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0
    keep = sizes >= min_size_px
    return keep[labeled]


def remove_small_holes_scipy(binary_mask, area_threshold_px, connectivity=2):
    inverted = ~binary_mask.astype(bool)
    filled_inverted = remove_small_objects_scipy(inverted, min_size_px=area_threshold_px, connectivity=connectivity)
    return ~filled_inverted


def full_cleanup_pipeline_protected(pred_opt, min_feature_size_px, protect_mask=None,
                                     threshold=0.5, distance_between_elements=1.0):
    """
    protect_mask: bool/float array matching pred_opt's shape; True/1 marks pixels
    that must not change (e.g. the waveguide region). Original pred_opt values
    are kept there.
    """
    is_torch = hasattr(pred_opt, 'detach')
    pred_opt_np = pred_opt.detach().cpu().numpy() if is_torch else np.asarray(pred_opt, dtype=np.float64)

    orig_shape = pred_opt_np.shape
    pred_opt_2d = np.squeeze(pred_opt_np)

    if protect_mask is not None:
        if hasattr(protect_mask, 'detach'):
            protect_mask = protect_mask.detach().cpu().numpy()
        protect_mask_2d = np.squeeze(np.asarray(protect_mask)).astype(bool)
    else:
        protect_mask_2d = np.zeros_like(pred_opt_2d, dtype=bool)

    r_filter_px = max(1.0, min_feature_size_px / 2.0)

    smoothed = circular_density_filter_np(pred_opt_2d, r_filter=r_filter_px,
                                           distance_between_elements=distance_between_elements)
    binary = smoothed > threshold
    min_area = np.pi * r_filter_px**2

    cleaned = remove_small_objects_scipy(binary, min_size_px=min_area)
    cleaned = remove_small_holes_scipy(cleaned, area_threshold_px=min_area)
    cleaned = cleaned.astype(np.float64)

    # protected region (waveguide) keeps its original pred_opt values
    cleaned[protect_mask_2d] = pred_opt_2d[protect_mask_2d]

    cleaned = cleaned.reshape(orig_shape)

    if is_torch:
        cleaned = torch.as_tensor(cleaned, dtype=pred_opt.dtype, device=pred_opt.device)
    return cleaned

In [ ]:
H, W = optimized_sup_map.shape[-2], optimized_sup_map.shape[-1]
mask_waveguide = torch.ones(1, 1, H, W)
c = H // 2
mask_waveguide[0, 0, c - 40 : c + 40, :] = 0

protect_mask = (mask_waveguide == 0)  # True where the waveguide is — keep unchanged

optimized_sup_map_f = full_cleanup_pipeline_protected(
    optimized_sup_map,
    min_feature_size_px=8,
    protect_mask=protect_mask
)

## Save the final binary support map

In [ ]:
plt.imshow(optimized_sup_map_f)

In [ ]:
np.savez_compressed('optimized_support_map_final.npz', array=optimized_sup_map_f)